In [5]:
import pandas as pd
import xarray as xr
import numpy as np
import os

# ========== 配置 ==========
BASE_DIR = r"F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures"
CSV_PATH = os.path.join(BASE_DIR, "195004allfeature.csv")
NC_PATH = os.path.join(BASE_DIR, "..", "ProcessFile", "snow3para.nc")
OUTPUT_PATH = os.path.join(BASE_DIR, "195004allfeature_with_snowc_sde.csv")

# ========== 加载数据 ==========
print("🔄 加载 CSV 文件...")
df = pd.read_csv(CSV_PATH)

print("🔄 加载 NetCDF 文件...")
ds = xr.open_dataset(NC_PATH)
times = pd.to_datetime(ds['valid_time'].values)

# ========== 查找目标时间 ==========
target_year = 1950
target_month = 4

matched_idx = None
for i, t in enumerate(times):
    if t.year == target_year and t.month == target_month:
        matched_idx = i
        break

if matched_idx is None:
    raise ValueError("❌ 没有找到匹配的时间点（1950年4月）")
print(f"📌 使用时间索引 {matched_idx}（{times[matched_idx]}）")

# ========== 提取变量 ==========
snowc = ds['snowc'].isel(valid_time=matched_idx).values
sde = ds['sde'].isel(valid_time=matched_idx).values
lat_grid = ds['latitude'].values
lon_grid = ds['longitude'].values

# ========== 转换 NetCDF 经度格式 ==========
def convert_lon(lon):
    """将 0-360 经度转换为 -180-180 经度"""
    return lon - 360 if lon > 180 else lon

# 转换经度
lon_grid_converted = np.vectorize(convert_lon)(lon_grid)

# ========== 查找经纬度并添加变量 ==========
def find_nearest_idx(array, value):
    return np.abs(array - value).argmin()

# 添加 snowc 和 sde 到 CSV 文件
snowc_list = []
sde_list = []

for i, row in df.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    
    # 查找最接近的经纬度索引
    lat_idx = find_nearest_idx(lat_grid, lat)
    lon_idx = find_nearest_idx(lon_grid_converted, lon)
    
    # 获取 snowc 和 sde 值
    snowc_val = snowc[lat_idx, lon_idx]
    sde_val = sde[lat_idx, lon_idx]
    
    snowc_list.append(snowc_val)
    sde_list.append(sde_val)

# ========== 将结果添加到 DataFrame 并保存 ==========
df['snowc'] = snowc_list
df['sde'] = sde_list

# 保存新的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ 结果已保存到 {OUTPUT_PATH}")


🔄 加载 CSV 文件...
🔄 加载 NetCDF 文件...
📌 使用时间索引 3（1950-04-01 00:00:00）
✅ 结果已保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\195004allfeature_with_snowc_sde.csv


In [3]:
import pandas as pd
import xarray as xr
import numpy as np
import os

# ========== 配置 ==========
BASE_DIR = r"F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures"
NC_PATH = os.path.join(BASE_DIR, "..", "ProcessFile", "snow3para.nc")
OUTPUT_DIR = os.path.join(BASE_DIR, "ProcessedCSV")  # 输出目录

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========== 加载 NetCDF 文件 ==========
print("🔄 加载 NetCDF 文件...")
ds = xr.open_dataset(NC_PATH)
times = pd.to_datetime(ds['valid_time'].values)

# ========== 转换 NetCDF 经度格式 ==========
def convert_lon(lon):
    """将 0-360 经度转换为 -180-180 经度"""
    return lon - 360 if lon > 180 else lon

# 获取经纬度数据
lat_grid = ds['latitude'].values
lon_grid = ds['longitude'].values
snowc = ds['snowc'].values
sde = ds['sde'].values
lon_grid_converted = np.vectorize(convert_lon)(lon_grid)

# ========== 查找目标时间 ==========
def find_nearest_idx(array, value):
    """查找最接近的经纬度索引"""
    return np.abs(array - value).argmin()

# ========== 处理文件夹中的所有 CSV 文件 ==========
for year in range(1951, 1952):
    for month in range(7, 10):
        print(f"🔄 处理 {year}年{month}月...")

        # 构建对应的 CSV 文件路径
        csv_filename = f"{year}{month:02d}allfeature.csv"
        csv_filepath = os.path.join(BASE_DIR, csv_filename)

        # 检查 CSV 文件是否存在
        if not os.path.exists(csv_filepath):
            print(f"❌ 文件 {csv_filename} 不存在，跳过...")
            continue

        # 加载 CSV 文件
        df = pd.read_csv(csv_filepath)

        # 查找匹配的时间索引
        matched_idx = None
        for i, t in enumerate(times):
            if t.year == year and t.month == month:
                matched_idx = i
                break

        if matched_idx is None:
            print(f"❌ 没有找到匹配的时间点 ({year}-{month})")
            continue

        print(f"📌 使用时间索引 {matched_idx}（{times[matched_idx]}）")

        # 提取该月的 snowc 和 sde 数据
        snowc_month = snowc[matched_idx]
        sde_month = sde[matched_idx]

        # 添加 snowc 和 sde 到 CSV 文件
        snowc_list = []
        sde_list = []

        for i, row in df.iterrows():
            lat = row['latitude']
            lon = row['longitude']

            # 查找最接近的经纬度索引
            lat_idx = find_nearest_idx(lat_grid, lat)
            lon_idx = find_nearest_idx(lon_grid_converted, lon)

            # 获取 snowc 和 sde 值
            snowc_val = snowc_month[lat_idx, lon_idx]
            sde_val = sde_month[lat_idx, lon_idx]

            snowc_list.append(snowc_val)
            sde_list.append(sde_val)

        # 将结果添加到 DataFrame 并保存
        df['snowc'] = snowc_list
        df['sde'] = sde_list

        # 保存到新的 CSV 文件
        output_filepath = os.path.join(OUTPUT_DIR, csv_filename)
        df.to_csv(output_filepath, index=False)
        print(f"✅ 已处理并保存到 {output_filepath}")


🔄 加载 NetCDF 文件...
🔄 处理 1951年7月...
📌 使用时间索引 18（1951-07-01 00:00:00）
✅ 已处理并保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\ProcessedCSV\195107allfeature.csv
🔄 处理 1951年8月...
📌 使用时间索引 19（1951-08-01 00:00:00）
✅ 已处理并保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\ProcessedCSV\195108allfeature.csv
🔄 处理 1951年9月...
📌 使用时间索引 20（1951-09-01 00:00:00）
✅ 已处理并保存到 F:\TUB\greenlandProject\ERA5LAND\MonthlyFeatures\ProcessedCSV\195109allfeature.csv
